# Data Cleaning -- High PDL1 1L Pembrolizumab + Chemotherapy vs. Pembrolizumab
**This notebook prepares Flatiron Health CSV files for patients with advanced non-small cell lung cancer with high PDL1 (>= 50%) treated with first-line pembrolizumab plus platinum-based chemotherapy or pembrolizumab. Patients that are EGFR or ALK positive are excluded. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorNSCLC
from flatiron_cleaner import merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
df.head(3)

,PatientID,LineName,StartDate
0,FA2BFA6104994,pembro_platinum,2019-08-19
1,F40B0A967B8AC,pembro_platinum,2020-03-13
2,F242D29EFD1F3,pembro_platinum,2024-03-08


In [4]:
df.shape

(20623, 3)

In [5]:
ids = df.PatientID.to_list()

In [6]:
len(ids)

20623

## Clean CSV files 

In [7]:
# Initialize class 
processor = DataProcessorNSCLC()

### Process Enhanced_AdvancedNSCLC.csv

In [8]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_AdvancedNSCLC.csv',
                                         patient_ids = ids,
                                         drop_dates = False)

2026-03-16 13:23:02,810 - INFO - Successfully read Enhanced_AdvancedNSCLC.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 13:23:02,811 - INFO - Filtering for 20623 specific PatientIDs
2026-03-16 13:23:02,823 - INFO - Successfully filtered Enhanced_AdvancedNSCLC.csv file with shape: (20623, 6) and unique PatientIDs: 20623
2026-03-16 13:23:02,838 - INFO - Successfully processed Enhanced_AdvancedNSCLC.csv file with final shape: (20623, 8) and unique PatientIDs: 20623


In [9]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

enhanced_df = pd.merge(enhanced_df, df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')
enhanced_df['days_adv_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['AdvancedDiagnosisDate']).dt.days
enhanced_df['days_adv_to_treatment'] = np.where(enhanced_df['days_adv_to_treatment'] < 0 , 0, enhanced_df['days_adv_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_adv_to_treatment'] < 30, 1, 0)

In [10]:
enhanced_df['days_diagnosis_to_adv'] = np.where(enhanced_df['days_diagnosis_to_adv'] < 0 , 0, enhanced_df['days_diagnosis_to_adv'])
enhanced_df['days_diagnosis_to_adv'] = enhanced_df['days_diagnosis_to_adv'].fillna(0)

In [11]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         15298
III         2262
I           1825
II           820
unknown      417
0              1
Name: count, dtype: int64

In [12]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [13]:
enhanced_df['adv_diagnosis_year'] = pd.to_numeric(enhanced_df['adv_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['adv_diagnosis_year'] < 2020, 1, 0)

In [14]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'AdvancedDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [15]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2026-03-16 13:23:02,937 - INFO - Successfully read Demographics.csv file with shape: (114756, 6) and unique PatientIDs: 114756
2026-03-16 13:23:02,975 - INFO - Successfully processed Demographics.csv file with final shape: (20623, 6) and unique PatientIDs: 20623


In [16]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M    11082
F     9541
Name: count, dtype: int64

In [17]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [18]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_AdvNSCLCBiomarkers.csv

In [19]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-16 13:23:04,037 - INFO - Successfully read Enhanced_AdvNSCLCBiomarkers.csv file with shape: (1021678, 18) and unique PatientIDs: 93715
2026-03-16 13:23:04,348 - INFO - Successfully merged Enhanced_AdvNSCLCBiomarkers.csv df with index_date_df resulting in shape: (273395, 19) and unique PatientIDs: 20074
2026-03-16 13:23:05,682 - INFO - Successfully processed Enhanced_AdvNSCLCBiomarkers.csv file with final shape: (20623, 11) and unique PatientIDs: 20623


In [20]:
biomarkers_df['PDL1_percent_staining'] = biomarkers_df["PDL1_percent_staining"].map({
    '0%': '0%', 
    '<1%': '1-49%',
    '1%': '1-49%',
    '2% - 4%': '1-49%',
    '5% - 9%': '1-49%', 
    '10% - 19%': '1-49%',
    '20% - 29%': '1-49%',
    '30% - 39%': '1-49%',
    '40% - 49%': '1-49%',
    '50% - 59%': '>=50%', 
    '60% - 69%': '>=50%', 
    '70% - 79%': '>=50%',
    '80% - 89%': '>=50%', 
    '90% - 99%': '>=50%', 
    '100%': '>=50%',
})

biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].fillna('unknown')
biomarkers_df['PDL1_percent_staining'] = biomarkers_df['PDL1_percent_staining'].astype('category')

In [21]:
biomarkers_df.PDL1_percent_staining.value_counts(dropna = False)

PDL1_percent_staining
unknown    16982
>=50%       2118
1-49%       1522
0%             1
Name: count, dtype: int64

In [22]:
biomarkers_df['EGFR_status'] = biomarkers_df['EGFR_status'].fillna('unknown')
biomarkers_df['ALK_status'] = biomarkers_df['ALK_status'].fillna('unknown')

In [23]:
biomarkers_df.EGFR_status.value_counts(dropna = False)

EGFR_status
negative    15991
unknown      4295
positive      337
Name: count, dtype: int64

In [24]:
biomarkers_df.ALK_status.value_counts(dropna = False)

ALK_status
negative    16092
unknown      4462
positive       69
Name: count, dtype: int64

### Process ECOG.csv

In [25]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-16 13:23:06,242 - INFO - Successfully read ECOG.csv file with shape: (1623197, 4) and unique PatientIDs: 85577
2026-03-16 13:23:06,493 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (390218, 5) and unique PatientIDs: 17768
2026-03-16 13:23:06,791 - INFO - Successfully processed ECOG.csv file with final shape: (20623, 3) and unique PatientIDs: 20623


In [26]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
1      7255
NaN    4930
0      4636
2      2998
3       754
4        50
Name: count, dtype: int64

In [27]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 1 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(1)

In [28]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [29]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-16 13:24:17,082 - INFO - Successfully read Vitals.csv file with shape: (29862758, 16) and unique PatientIDs: 114285
2026-03-16 13:24:27,722 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (6486234, 17) and unique PatientIDs: 20623
2026-03-16 13:24:30,660 - INFO - Successfully processed Vitals.csv file with final shape: (20623, 8) and unique PatientIDs: 20623


### Process Lab.csv

In [30]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-16 13:26:30,456 - INFO - Successfully read Lab.csv file with shape: (82093498, 17) and unique PatientIDs: 108761
2026-03-16 13:26:54,210 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (17933203, 18) and unique PatientIDs: 20411
2026-03-16 13:27:33,168 - INFO - Successfully processed Lab.csv file with final shape: (20623, 76) and unique PatientIDs: 20623


### Process MedicationAdministration.csv

In [31]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-16 13:27:42,086 - INFO - Successfully read MedicationAdministration.csv file with shape: (7486734, 11) and unique PatientIDs: 89605
2026-03-16 13:27:43,919 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (1633665, 12) and unique PatientIDs: 20038
2026-03-16 13:27:44,056 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (20623, 9) and unique PatientIDs: 20623


### Process Diagnosis.csv

In [32]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-16 13:27:49,607 - INFO - Successfully read Diagnosis.csv file with shape: (9083016, 6) and unique PatientIDs: 114756
2026-03-16 13:27:50,771 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (1991348, 7) and unique PatientIDs: 20623
2026-03-16 13:27:55,503 - INFO - Successfully processed Diagnosis.csv file with final shape: (20623, 39) and unique PatientIDs: 20623


### Process Enhanced_Mortality_V2.csv

In [33]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-03-16 13:27:55,587 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-03-16 13:27:55,642 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (20623, 3) and unique PatientIDs: 20623
2026-03-16 13:27:59,656 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-16 13:27:59,668 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (20623, 6) and unique PatientIDs: 20623. There are 0 out of 20623 patients with missing duration values


In [34]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [35]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [36]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

/Users/xavierorcutt/Dropbox/TrialRescue/myenv/lib/python3.13/site-packages/flatiron_cleaner/general.py:1363: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
2026-03-16 13:28:00,250 - INFO - Successfully read Insurance.csv file with shape: (383622, 14) and unique PatientIDs: 106441
2026-03-16 13:28:00,430 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (74975, 15) and unique PatientIDs: 19890
2026-03-16 13:28:00,608 - INFO - Successfully processed Insurance.csv file with final shape: (20623, 5) and unique PatientIDs: 20623


### Process SocialDeterminantsOfHealth.csv 

In [37]:
ses_df = pd.read_csv('../data/SocialDeterminantsOfHealth.csv')

In [38]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,F9F11E927B585,5 - Highest SES
1,F998F69458EED,5 - Highest SES
2,F05B7A9AEA213,NaN


In [39]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
4                  22426
3                  21632
2                  20561
5 - Highest SES    18784
1 - Lowest SES     18593
NaN                11618
Name: count, dtype: int64

In [40]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 3 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(3)

In [41]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [42]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

## Merge dataframes

In [43]:
pembrochemo_pembro_features_df = merge_dataframes(enhanced_df,
                                                  demographics_df,
                                                  biomarkers_df,
                                                  ecog_df,
                                                  vitals_df,
                                                  labs_df,
                                                  medications_df,
                                                  diagnosis_df, 
                                                  mortality_df, 
                                                  insurance_df,
                                                  ses_df,
                                                  merge_type = 'inner')

2026-03-16 13:28:00,690 - INFO - Anticipated number of merges: 10
2026-03-16 13:28:00,692 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 164
2026-03-16 13:28:00,715 - INFO - Dataset 1 shape: (20623, 10), unique PatientIDs: 20623
2026-03-16 13:28:00,724 - INFO - Dataset 2 shape: (20623, 6), unique PatientIDs: 20623
2026-03-16 13:28:00,733 - INFO - Dataset 3 shape: (20623, 11), unique PatientIDs: 20623
2026-03-16 13:28:00,736 - INFO - Dataset 4 shape: (20623, 4), unique PatientIDs: 20623
2026-03-16 13:28:00,739 - INFO - Dataset 5 shape: (20623, 8), unique PatientIDs: 20623
2026-03-16 13:28:00,741 - INFO - Dataset 6 shape: (20623, 76), unique PatientIDs: 20623
2026-03-16 13:28:00,743 - INFO - Dataset 7 shape: (20623, 9), unique PatientIDs: 20623
2026-03-16 13:28:00,745 - INFO - Dataset 8 shape: (20623, 39), unique PatientIDs: 20623
2026-03-16 13:28:00,747 - INFO - Dataset 9 shape: (20459, 3), unique PatientIDs: 20459
2026-0

In [44]:
pembrochemo_pembro_features_df.shape

(20316, 164)

In [45]:
pembrochemo_pembro_features_df.head(2)

,PatientID,Histology,SmokingStatus,GroupStage_mod,days_diagnosis_to_adv,adv_diagnosis_year,days_adv_to_treatment,days_to_treatment_before_30d,GroupStage_mod_na,before_2020,...,other_viscera_met,other_met,event,duration,commercial,medicaid,medicare,other_insurance,ses_mod,ses_mod_na
0,FA2BFA6104994,Non-squamous cell carcinoma,History of smoking,1.0,4741.0,2017,703,0,0,1,...,0,0,0,2256.0,1,0,1,1,3.0,0
1,F40B0A967B8AC,Non-squamous cell carcinoma,History of smoking,4.0,0.0,2020,17,1,0,0,...,0,0,1,946.0,0,0,1,0,4.0,0


In [46]:
pembrochemo_pembro_features_df = pembrochemo_pembro_features_df.query('PDL1_percent_staining == ">=50%"')
pembrochemo_pembro_features_df = pembrochemo_pembro_features_df.query('EGFR_status == "negative" or EGFR_status == "unknown"')
pembrochemo_pembro_features_df = pembrochemo_pembro_features_df.query('ALK_status == "negative" or ALK_status == "unknown"')

In [47]:
pembrochemo_pembro_features_df.shape

(2064, 164)

## Export dataframe

In [48]:
pembrochemo_pembro_features_df.to_csv('../outputs/pembrochemo_pembro_features_df.csv', index = False)

In [49]:
# Save dtypes
pembrochemo_pembro_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/pembrochemo_pembro_features_dtypes.csv')